In [ ]:
%pip install -r ./requirements.txt

In [11]:
### Q1

import re
from pathlib import Path
import requests
import pandas as pd


BOOKS: dict[int, str] = {
    2701: "Moby Dick",
    1342: "Pride and Prejudice",
    84: "Frankenstein",
    1661: "The Adventures of Sherlock Holmes",
    11: "Alice's Adventures in Wonderland",
    74: "The Adventures of Tom Sawyer",
    98: "A Tale of Two Cities",
    1232: "The Prince",
    5200: "Metamorphosis",
    1952: "The Yellow Wallpaper",
}

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

def download_book(book_id: int) -> str | None:
    url: str = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
    response = requests.get(url = url, timeout=20)
    if response.status_code == 200:
        response.encoding = "utf-8"
        return response.text
    return None

def clean_book_text(book_name: str, book_text: str) -> str:
    start_pattern: str = fr"\*\*\* START OF THE PROJECT GUTENBERG EBOOK {book_name.upper()}.*?\*\*\*"
    end_pattern: str = fr"\*\*\* END OF THE PROJECT GUTENBERG EBOOK {book_name.upper()}.*?\*\*\*"
    book_text = re.sub(r"\r\n", "\n", book_text) # replaces all windows-style line-endings to unix-like
    start_match: re.Match[str] | None = re.search(start_pattern, book_text, flags=re.IGNORECASE | re.DOTALL)
    end_match: re.Match[str] | None = re.search(end_pattern, book_text, flags=re.IGNORECASE | re.DOTALL)

    start_index = start_match.end() if start_match is not None else 0
    end_index = end_match.start() if end_match is not None else len(book_text)
    book_text = book_text[start_index:end_index]
    book_text = re.sub(r"\s+", " ", book_text)
    return book_text.strip()


DOWNLOAD_BASE_URL = 'www.gutenberg.org/ebooks'
def download_books():
    for book_id, book_name in BOOKS.items():
        result = download_book(book_id=book_id)
        if result is None:
            raise ValueError(f"failure during downloading of book {book_id}")
        text = clean_book_text(book_name=book_name, book_text=result)
        path = DATA_DIR / f"{book_id}.txt"
        path.write_text(data = text, encoding="utf-8")
        print(f"Book: {book_name} with id: {book_id}. File Size = {round(path.stat().st_size / 1024, 2)}kB. Beggining = {text[:100].replace("\n", "")}")
        

download_books()

Book: Moby Dick with id: 2701. File Size = 1201.68kB. Beggining = MOBY-DICK; or, THE WHALE. By Herman Melville CONTENTS ETYMOLOGY. EXTRACTS (Supplied by a Sub-Sub-Lib
Book: Pride and Prejudice with id: 1342. File Size = 710.61kB. Beggining = [Illustration: GEORGE ALLEN PUBLISHER 156 CHARING CROSS ROAD LONDON RUSKIN HOUSE ] [Illustration: _R
Book: Frankenstein with id: 84. File Size = 410.63kB. Beggining = Frankenstein; or, the Modern Prometheus by Mary Wollstonecraft (Godwin) Shelley CONTENTS Letter 1 Le
Book: The Adventures of Sherlock Holmes with id: 1661. File Size = 559.38kB. Beggining = The Adventures of Sherlock Holmes by Arthur Conan Doyle Contents I. A Scandal in Bohemia II. The Red
Book: Alice's Adventures in Wonderland with id: 11. File Size = 146.12kB. Beggining = [Illustration] Alice’s Adventures in Wonderland by Lewis Carroll THE MILLENNIUM FULCRUM EDITION 3.0 
Book: The Adventures of Tom Sawyer with id: 74. File Size = 393.7kB. Beggining = THE ADVENTURES OF TOM SAWYER By 